# ValleyFever-NewsCast

## 2. Article Processing NLP

## Environment Setup

Run the cell below to set up the environment for either Google Colab or local execution:

In [57]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")

    # Clone repository if in Colab
    if not os.path.exists('/content/ValleyFever-NewsCast/'):
        !git clone https://github.com/Adrian1840/ValleyFever-NewsCast
    os.chdir('/content/ValleyFever-NewsCast')

except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add src directory to Python path
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Running in Google Colab
Current working directory: /content/ValleyFever-NewsCast


## Import Libraries and Data Loading

In [ ]:
  # Basic Packages
  import numpy as np
  import pandas as pd

  import re #Reg Expression NLP package

In [ ]:
article_pth = "/content/ValleyFever-NewsCast/data/raw/google_news_with_text.csv"
google_news_with_text = pd.read_csv(article_pth)

In [ ]:
google_news_with_text.head(2)

,Year-Month,query_term,title,source,real_url,article_text
0,2008-04,"""valley fever""",Llamas find refuge at Queen Creek facility - E...,East Valley Tribune,https://www.eastvalleytribune.com/news/llamas-...,"llamas are curious, expressive and polite anim..."
1,2008-08,"""valley fever""",Anthrax investigation generates valuable foren...,The NAU Review,https://news.nau.edu/anthrax-investigation-gen...,northern arizona university’s paul keim says t...


In [69]:
count_path = "/content/ValleyFever-NewsCast/data/raw/article_counts_only.csv"
article_counts_only = pd.read_csv(count_path)

In [70]:
article_counts_only.head()

,Year-Month,Num_Articles
0,2008-07,0
1,2008-08,1
2,2008-09,2
3,2008-10,0
4,2008-11,2


## Article Cleaning using NLP Techniques

We can clean and normalize the **article text** for each article in the `google_news_with_text` dataframe.


In [71]:
def clean_article_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove bracket content like [caption], [1], etc.
    text = re.sub(r"\[.*?\]", "", text)

    # remove parentheses with citations (1), (2), etc.
    text = re.sub(r"\(\d+\)", "", text)

    # remove source names like abcnews.com
    text = re.sub(r"\b[a-z]+\.(com|org|net)\b", "", text)

    # remove non-letter characters (keep spaces)
    text = re.sub(r"[^a-z\s]", " ", text)

    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [72]:
google_news_with_text["article_text_clean"] = google_news_with_text["article_text"].apply(clean_article_text)
google_news_with_text.head(3)

,Year-Month,query_term,title,source,real_url,article_text,article_text_clean,clean_length,risk_mentions,cv_mention_count
0,2008-04,"""valley fever""",Llamas find refuge at Queen Creek facility - E...,East Valley Tribune,https://www.eastvalleytribune.com/news/llamas-...,"llamas are curious, expressive and polite anim...",llamas are curious expressive and polite anima...,3386,0,0
1,2008-08,"""valley fever""",Anthrax investigation generates valuable foren...,The NAU Review,https://news.nau.edu/anthrax-investigation-gen...,northern arizona university’s paul keim says t...,northern arizona university s paul keim says t...,3886,3,0
2,2008-09,"""valley fever""","Black Box Warning for Remicade, Enbrel and Oth...",AboutLawsuits.com,https://www.aboutlawsuits.com/black-box-warnin...,add your comments\n\nthe fda has issued an ale...,add your comments the fda has issued an alert ...,3656,4,0


Removing empty `article_text_clean` rows:

In [73]:
google_news_with_text["clean_length"] = google_news_with_text["article_text_clean"].str.len()

google_news_with_text = google_news_with_text[google_news_with_text["clean_length"] > 0]

## Extracting Media-Related Features

### Risk Mentions

In [74]:
google_news_with_text["risk_mentions"] = google_news_with_text["article_text_clean"].str.count(
    r"\b(outbreak|surge|spike|increase|danger|severe|deadly|hospital|death)\b",
    flags=re.IGNORECASE
)

### Central Valley (CV) Mentions

In [75]:
google_news_with_text["cv_mention_count"] = google_news_with_text["article_text_clean"].str.count(
    r"\b(fresno|kern|tulare|kings|madera|merced|central valley)\b",
    flags=re.IGNORECASE
)

### Aggregating Article/Title Sentinment and Mentions by Month

Since there can be multiple articles for one month, we need to ***aggregate sentiment scores*** by month to properly merge with the previous news article data frame.

In [76]:
monthly_aggregates = (
    google_news_with_text.groupby("Year-Month")
    .agg(
        risk_mentions_count=("risk_mentions", "sum"),
        cv_mentions_count=("cv_mention_count", "sum"),
        article_count=("title", "size"), #helps with cleaning (checking for months with no articles)

   )
    .reset_index()
)

In [ ]:
monthly_aggregates.head(5)

# Converting Mention Counts to Mention Rates

In [79]:
monthly_aggregates["cv_mentions_rate"] = np.where(
    monthly_aggregates["article_count"] > 0,
    monthly_aggregates["cv_mentions_count"] / monthly_aggregates["article_count"],
    0
)

monthly_aggregates["risk_mentions_rate"] = np.where(
    monthly_aggregates["article_count"] > 0,
    monthly_aggregates["cv_mentions_count"] / monthly_aggregates["article_count"],
    0
)

In [ ]:
monthly_aggregates

,Year-Month,risk_mentions_count,cv_mentions_count,article_count,cv_mentions_rate,risk_mentions_rate
0,2008-04,0,0,1,0.000000,0.000000
1,2008-08,3,0,1,0.000000,0.000000
2,2008-09,5,0,2,0.000000,0.000000
3,2008-10,0,0,1,0.000000,0.000000
4,2008-11,0,0,1,0.000000,0.000000
...,...,...,...,...,...,...
66,2015-08,19,1,6,0.166667,0.166667
67,2015-09,18,0,11,0.000000,0.000000
68,2015-10,12,2,6,0.333333,0.333333
69,2015-11,18,0,10,0.000000,0.000000


There is also going to be months with no articles, so lets replace those article count columns for these months with 0.

In [80]:
# Complete monthly range
months = pd.DataFrame({"Year-Month": pd.period_range("2008-07","2015-12",freq="M").astype(str)})

# Merge with aggregated counts and fill NAs with 0
monthly_aggregates = (months.merge(monthly_aggregates, on="Year-Month", how="left").fillna(0))

monthly_aggregates

,Year-Month,risk_mentions_count,cv_mentions_count,article_count,cv_mentions_rate,risk_mentions_rate
0,2008-07,0.0,0.0,0.0,0.000000,0.000000
1,2008-08,3.0,0.0,1.0,0.000000,0.000000
2,2008-09,5.0,0.0,2.0,0.000000,0.000000
3,2008-10,0.0,0.0,1.0,0.000000,0.000000
4,2008-11,0.0,0.0,1.0,0.000000,0.000000
...,...,...,...,...,...,...
85,2015-08,19.0,1.0,6.0,0.166667,0.166667
86,2015-09,18.0,0.0,11.0,0.000000,0.000000
87,2015-10,12.0,2.0,6.0,0.333333,0.333333
88,2015-11,18.0,0.0,10.0,0.000000,0.000000


## Creating Processed News Features Dataset

In [81]:
news_features_unlagged = article_counts_only.merge(
    monthly_aggregates[
        ["Year-Month", "cv_mentions_rate", "risk_mentions_rate"]
    ],
    on="Year-Month",
    how="left"
)

news_features_unlagged.head()

,Year-Month,Num_Articles,cv_mentions_rate,risk_mentions_rate
0,2008-07,0,0.0,0.0
1,2008-08,1,0.0,0.0
2,2008-09,2,0.0,0.0
3,2008-10,0,0.0,0.0
4,2008-11,2,0.0,0.0


In [82]:
news_features_unlagged.to_csv("data/processed/news_features_unlagged.csv", index=False)